# 100_100の学習
既存のtrain_L1_normalization.ipynbを複製し、隠れ層を100・100へ変更。保存先から実行すること。学習入力・正解・学習率・反復数・L1係数は共通。シード0を固定する。


In [1]:
# !pip install torch numpy matplotlib

In [2]:
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

### PPMのRGBを正規化し、現在の学習処理に合わせた33画素に変換する

実験フォルダ内の`../scanner_ver2/before_1669_59.ppm`（新しい3×3入力）と、ノートブックと同じフォルダの`week3_6_6.ppm`（従来の6×6入力）を読み込み、RGBの各成分を255で割って0.0～1.0に変換します。3×3の9画素と、6×6の上4行の24画素をこの順番で結合します。結果の`X_ppm`は`(33, 3)`の`float32`配列です。

現在の正解データ`Y`は3×3と6×4の色見本から作られます。この変換は、6×6の上4行が6×4の色見本に対応することを前提にしています。下2行の12画素は使いません。各画像の画素順は左上から行ごとです。実際の読み取り位置と正解の色見本が対応しているかも確認してください。

下の元のデータ設定セルの後に追加したセルで、`X = X_ppm.copy()`を実行します。元のサンプルデータを変換結果で上書きし、`X`と`Y`の形状を確認してから学習に渡します。上から順に実行してください。変換セルは単独でも実行でき、`X = np.array([...], dtype=np.float32)`の形式で結果を出力します。


In [3]:
from pathlib import Path
import numpy as np

# 3×3は今回取得した補正前データ。6×6は従来データを使用する。
file_3x3 = "../scanner_ver2/before_1669_59.ppm"
file_6x6 = "week3_6_6.ppm"

# ノートブックと同じフォルダに置いたPPMファイルを読み込む。
ppm_dir = Path("../..")


def read_normalized_ppm(path, expected_size):
    """P3形式のPPMを、左上から行ごとに並べたRGB配列に変換する。"""
    text = path.read_text(encoding="utf-8")
    tokens = " ".join(line.split("#", 1)[0] for line in text.splitlines()).split()
    if len(tokens) < 4 or tokens[0] != "P3":
        raise ValueError(f"{path.name}: P3形式のPPMを指定してください。")

    width, height, max_value = map(int, tokens[1:4])
    if (width, height) != expected_size or max_value != 255:
        raise ValueError(f"{path.name}: サイズ{expected_size}、最大値255が必要です。")

    values = np.array([int(value) for value in tokens[4:]], dtype=np.float32)
    if values.size != width * height * 3:
        raise ValueError(f"{path.name}: RGB値の数が画像サイズと一致しません。")
    if not np.all((values >= 0) & (values <= 255)):
        raise ValueError(f"{path.name}: RGB値は0～255である必要があります。")

    return values.reshape(-1, 3) / 255.0


X_3x3 = read_normalized_ppm(ppm_dir / file_3x3, (3, 3))
X_6x6 = read_normalized_ppm(ppm_dir / file_6x6, (6, 6))

# 正解データimg2は6×4なので、6×6のPPMから上4行を取り出す。
# 上4行が6×4の色見本に対応する読み取り順を前提とする。
# 下2行の12画素は学習に使用しない。
X_6x4 = X_6x6.reshape(6, 6, 3)[:4, :, :].reshape(24, 3)

# 正解データYと同じ順番：3×3の9画素 → 6×4の24画素。
X_ppm = np.concatenate([X_3x3, X_6x4], axis=0).astype(np.float32)

print("3×3:", X_3x3.shape, "6×4:", X_6x4.shape, "学習用:", X_ppm.shape)
print("\n# 以下を学習用Xの定義として使用できます。")
print("X = np.array([")
for r, g, b in X_ppm:
    print(f"    [{r:.8f}, {g:.8f}, {b:.8f}],")
print("], dtype=np.float32)")


3×3: (9, 3) 6×4: (24, 3) 学習用: (33, 3)

# 以下を学習用Xの定義として使用できます。
X = np.array([
    [0.70980394, 0.20392157, 0.08235294],
    [0.23921569, 0.98039216, 0.21176471],
    [0.07058824, 0.25490198, 0.37647060],
    [0.88627452, 0.99215686, 0.26666668],
    [0.26274511, 0.98431373, 0.84705883],
    [0.41176471, 0.23921569, 0.31764707],
    [0.00392157, 0.00784314, 0.00784314],
    [0.20784314, 0.46666667, 0.26666668],
    [0.98431373, 1.00000000, 0.99215686],
    [0.22745098, 0.23529412, 0.10196079],
    [0.81960785, 0.75294119, 0.33725491],
    [0.22745098, 0.56862748, 0.50980395],
    [0.17647059, 0.40784314, 0.12941177],
    [0.38823530, 0.60000002, 0.61176473],
    [0.39215687, 0.98823529, 0.70980394],
    [1.00000000, 0.53725493, 0.13725491],
    [0.16470589, 0.34509805, 0.53725493],
    [0.78039217, 0.29411766, 0.19607843],
    [0.18431373, 0.16078432, 0.19607843],
    [0.59607846, 0.98431373, 0.23921569],
    [1.00000000, 0.96470588, 0.18823530],
    [0.04705882, 0.22745098, 0.37647060],

In [4]:
# 推論データ
X =  np.array([[0.77, 0.10, 0.06],
               [0.22, 0.86, 0.17],
               [0.95, 0.19, 0.31],
               [0.95, 1.00, 0.24],
               [0.29, 0.97, 0.75],
               [0.39, 0.15, 0.24],
               [0.00, 0.00, 0.00],
               [0.12, 0.25, 0.15],
               [1.00, 1.00, 1.00],

               [0.11, 0.04, 0.00],
               [0.41, 0.41, 0.19],
               [0.10, 0.29, 0.27],
               [0.05, 0.15, 0.03],
               [0.19, 0.35, 0.35],
               [0.25, 0.85, 0.44],

               [0.55, 0.29, 0.05],
               [0.05, 0.18, 0.30],
               [0.43, 0.13, 0.11],
               [0.05, 0.01, 0.08],
               [0.34, 0.80, 0.14],
               [0.66, 0.60, 0.12],

               [0.05, 0.08, 0.21],
               [0.08, 0.39, 0.06],
               [0.28, 0.00, 0.00],
               [0.63, 0.84, 0.11],
               [0.35, 0.10, 0.18],
               [0.06, 0.39, 0.34],

               [0.90, 1.00, 0.87],
               [0.56, 0.94, 0.55],
               [0.29, 0.49, 0.28],
               [0.12, 0.19, 0.11],
               [0.03, 0.05, 0.03],
               [0.00, 0.00, 0.00]], dtype=np.float32)

img1 = cv2.resize(cv2.imread("../../img/reference_image1_cmyk_large.png", cv2.IMREAD_COLOR_RGB), (3,3), interpolation=cv2.INTER_NEAREST)
img2 = cv2.resize(cv2.imread("../../img/reference_image2_cmyk_large.png", cv2.IMREAD_COLOR_RGB), (6,4), interpolation=cv2.INTER_NEAREST)

plt.imshow(img1)
plt.show()
plt.imshow(img2)
plt.show()

# 正解データ
Y = np.concatenate([img1.reshape([-1, 3])/255, img2.reshape([-1, 3])/255], axis=0).astype(np.float32)

In [5]:
# PPMから変換したデータを、学習用Xに設定する。
# このセルは、上のPPM変換セルと元の正解データYの設定セルの後に実行する。
X = X_ppm.copy()

if X.shape != Y.shape:
    raise ValueError(f"XとYの形状が一致しません: X={X.shape}, Y={Y.shape}")
if X.shape != (33, 3):
    raise ValueError(f"学習には33画素のRGBが必要です: X={X.shape}")
if not np.all(np.isfinite(X)) or not np.all((X >= 0.0) & (X <= 1.0)):
    raise ValueError("XのRGB値は0.0～1.0の有限な数値である必要があります。")

print(f"学習用データ: X={X.shape}, Y={Y.shape}, dtype={X.dtype}")
print("次の学習セルを実行できます。")


学習用データ: X=(33, 3), Y=(33, 3), dtype=float32
次の学習セルを実行できます。


In [6]:
# 再現用の乱数シードとCPU実行条件を記録する。
torch.manual_seed(0)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)

X_tensor = torch.tensor(X, dtype=torch.float32)
Y_tensor = torch.tensor(Y, dtype=torch.float32)

# モデル定義
class ColorNet(nn.Module):
    def __init__(self):
        super(ColorNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(3, 600),
            nn.ReLU(),
            nn.Linear(600, 600),
            nn.ReLU(),
            nn.Linear(600, 3),
        )

    def forward(self, x):
        return self.model(x)

model = ColorNet()

# 損失関数（MSE）
criterion = nn.MSELoss()

# 最適化手法
optimizer = optim.Adam(model.parameters(), lr=0.01)

# L1正則化の係数
lambda_l1 = 1e-6

# 学習ループ
epochs = 50000
for epoch in range(epochs):
    optimizer.zero_grad()

    outputs = model(X_tensor)
    mse = criterion(outputs, Y_tensor)

    # --- L1 正則化-----------------------------
    l1 = torch.tensor(0.0, requires_grad=False)
    for name, p in model.named_parameters():
        if 'weight' in name:
            l1 = l1 + p.abs().sum()
    loss = mse + lambda_l1 * l1
    # -----------------------------------------

    loss.backward()
    optimizer.step()

    if epoch % 500 == 0:
        print(f'Epoch [{epoch}/{epochs}]  MSE: {mse.item():.6f}  L1: {l1.item():.2f}  Loss: {loss.item():.6f}')

with torch.no_grad():
    predictions = model(X_tensor)
    print("\n予測結果：")
    print(predictions.numpy())

def convert_to_cpp_array(tensor: torch.Tensor, name: str, dtype: str = "float"):
    flat = tensor.detach().numpy().flatten()
    array_str = f"{dtype} {name}[] = {{"
    array_str += ", ".join(map(str, flat))
    array_str += "};"
    return array_str

cpp_code = ""

layer_idx = 1
for layer in model.model:
    if isinstance(layer, torch.nn.Linear):
        cpp_code += convert_to_cpp_array(layer.weight, f"weight_{layer_idx}") + "\n"
        cpp_code += convert_to_cpp_array(layer.bias,   f"bias_{layer_idx}") + "\n"
        layer_idx += 1

with open("model_parameters.h", "w") as f:
    f.write(cpp_code)

print("\nC++ 用のパラメータファイル (model_parameters.h) を作成しました。")


Epoch [0/50000]  MSE: 0.333865  L1: 7910.36  Loss: 0.341775
Epoch [500/50000]  MSE: 0.000350  L1: 6654.42  Loss: 0.007004
Epoch [1000/50000]  MSE: 0.000539  L1: 4729.68  Loss: 0.005269
Epoch [1500/50000]  MSE: 0.000182  L1: 3658.55  Loss: 0.003840
Epoch [2000/50000]  MSE: 0.000144  L1: 3029.10  Loss: 0.003173
Epoch [2500/50000]  MSE: 0.000201  L1: 2483.41  Loss: 0.002685
Epoch [3000/50000]  MSE: 0.000246  L1: 2037.58  Loss: 0.002284
Epoch [3500/50000]  MSE: 0.000128  L1: 1677.65  Loss: 0.001806
Epoch [4000/50000]  MSE: 0.000105  L1: 1372.09  Loss: 0.001477
Epoch [4500/50000]  MSE: 0.000116  L1: 1133.07  Loss: 0.001249
Epoch [5000/50000]  MSE: 0.000100  L1: 970.90  Loss: 0.001071
Epoch [5500/50000]  MSE: 0.000271  L1: 862.83  Loss: 0.001133
Epoch [6000/50000]  MSE: 0.000116  L1: 759.64  Loss: 0.000876
Epoch [6500/50000]  MSE: 0.000105  L1: 701.28  Loss: 0.000806
Epoch [7000/50000]  MSE: 0.000436  L1: 689.48  Loss: 0.001125
Epoch [7500/50000]  MSE: 0.000112  L1: 629.48  Loss: 0.000741
Ep